# Task 1.2: Pipeline Experimental de Auto-etiquetado con SAM

El objetivo es auto-generar las máscaras de segmentación iniciales de las tuberías submarinas de forma *zero-shot*, variando los hiperparámetros de densidad de puntos de muestreo sobre un lote controlado de 20 imágenes de prueba.






## 1. Configuración del entorno e instalación de SAM

In [1]:
# Instalación de dependencias
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q opencv-python-headless matplotlib numpy torch torchvision ipywidgets

import torch
print("Torch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (activa GPU en Colab: Entorno de ejecución > Cambiar tipo)")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 49.5 MB/s eta 0:00:00
Torch: 2.11.0+cu128
CUDA disponible: True
GPU: Tesla T4


## 2. Descarga de los pesos preentrenados

Para esta prueba use el mas ligero **ViT-B**.

In [2]:
import os, urllib.request

MODEL_TYPE = "vit_b"

CHECKPOINTS = {
    "vit_b": ("sam_vit_b_01ec64.pth", "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"),
    "vit_l": ("sam_vit_l_0b3195.pth", "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth"),
    "vit_h": ("sam_vit_h_4b8939.pth", "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"),
}

ckpt_name, ckpt_url = CHECKPOINTS[MODEL_TYPE]
if not os.path.exists(ckpt_name):
    print(f"Descargando {ckpt_name} ...")
    urllib.request.urlretrieve(ckpt_url, ckpt_name)
    print("Descarga completa.")
else:
    print(f"{ckpt_name} ya existe.")

USE_UNDERWATER_CHECKPOINT = False
UNDERWATER_CKPT_PATH = "watersam_vit_b.pth"

if USE_UNDERWATER_CHECKPOINT:
    assert os.path.exists(UNDERWATER_CKPT_PATH), "No se encontró el checkpoint submarino."
    ckpt_name = UNDERWATER_CKPT_PATH
    print(f"Usando checkpoint submarino: {ckpt_name}")

Descargando sam_vit_b_01ec64.pth ...
Descarga completa.


## 3. Carga del modelo SAM

In [3]:
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

sam = sam_model_registry[MODEL_TYPE](checkpoint=ckpt_name)
sam.to(device=DEVICE)
print(f"Modelo SAM ({MODEL_TYPE}) cargado en {DEVICE}.")

Modelo SAM (vit_b) cargado en cuda.


## 4. Carga del lote de imágenes

In [4]:
import glob, cv2
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/Under_Water_Dataset"
PROCESSED_DIR = os.path.join(DRIVE_ROOT, "Dataset_Processed")

FILTER_FOLDER = "Frames_CLAHE"
IMAGE_DIR = os.path.join(PROCESSED_DIR, FILTER_FOLDER)

# Tamaño del lote controlado
BATCH_SIZE = 20

exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff")
image_paths = []
for e in exts:
    image_paths += glob.glob(os.path.join(IMAGE_DIR, e))
# Excluir cualquier máscara previa p
image_paths = sorted([p for p in image_paths if "mask" not in os.path.basename(p).lower()])[:BATCH_SIZE]

if not image_paths:
    print(f" No hay imágenes en '{IMAGE_DIR}'.")
else:
    print(f"{len(image_paths)} imágenes cargadas desde '{FILTER_FOLDER}':")
    for p in image_paths[:5]:
        print("   -", os.path.basename(p))
    if len(image_paths) > 5:
        print(f"   ... (+{len(image_paths)-5} más)")

Mounted at /content/drive
20 imágenes cargadas desde 'Frames_CLAHE':
   - DeepWater-Video1_frame_0000.jpg
   - DeepWater-Video1_frame_0001.jpg
   - DeepWater-Video1_frame_0002.jpg
   - DeepWater-Video1_frame_0003.jpg
   - DeepWater-Video1_frame_0004.jpg
   ... (+15 más)


## 5. Experimento


In [5]:
def build_mask_generator(points_per_side=32,
                         pred_iou_thresh=0.86,
                         stability_score_thresh=0.92,
                         min_mask_region_area=100):
    return SamAutomaticMaskGenerator(
        model=sam,
        points_per_side=points_per_side,
        pred_iou_thresh=pred_iou_thresh,
        stability_score_thresh=stability_score_thresh,
        min_mask_region_area=min_mask_region_area,
    )

# Densidades a comparar en el experimento
DENSITIES = [16, 32, 64]
print("Densidades de muestreo a evaluar:", DENSITIES)

Densidades de muestreo a evaluar: [16, 32, 64]


### 5.1 Utilidades de máscara y visualización

In [6]:
def masks_to_binary(masks, shape):
    binary = np.zeros(shape[:2], dtype=np.uint8)
    for m in masks:
        binary[m["segmentation"]] = 255
    return binary

def load_rgb(path):
    img = cv2.imread(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

## 6. Generación de máscaras y visualización interactiva

Se generan las máscaras con la densidad elegida y se muestran 5 imágenes crudas junto a sus máscaras binarias.

In [7]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def run_and_show(points_per_side=32, n_show=5):
    if not image_paths:
        print("Sin imágenes cargadas.")
        return

    generator = build_mask_generator(points_per_side=points_per_side)
    subset = image_paths[:n_show]

    fig, axes = plt.subplots(len(subset), 2, figsize=(9, 4.2 * len(subset)))
    if len(subset) == 1:
        axes = axes.reshape(1, 2)

    for i, path in enumerate(subset):
        img = load_rgb(path)
        masks = generator.generate(img)
        binary = masks_to_binary(masks, img.shape)

        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"Cruda — {os.path.basename(path)}", fontsize=10)
        axes[i, 0].axis("off")

        axes[i, 1].imshow(binary, cmap="gray")
        axes[i, 1].set_title(f"Máscara SAM (pps={points_per_side}, {len(masks)} instancias)", fontsize=10)
        axes[i, 1].axis("off")

    plt.tight_layout()
    plt.show()


density_slider = widgets.SelectionSlider(
    options=DENSITIES, value=32,
    description="points_per_side:",
    style={"description_width": "initial"},
    continuous_update=False,
)
widgets.interact(run_and_show, points_per_side=density_slider, n_show=widgets.fixed(5));

interactive(children=(SelectionSlider(continuous_update=False, description='points_per_side:', index=1, option…

## 7. Exportar las máscaras auto-generadas

In [8]:

OUTPUT_DIR = os.path.join(PROCESSED_DIR, f"SAM_Masks_{FILTER_FOLDER}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

FINAL_DENSITY = 32   # densidad definitiva
final_gen = build_mask_generator(points_per_side=FINAL_DENSITY)

if image_paths:
    for path in image_paths:
        img = load_rgb(path)
        masks = final_gen.generate(img)
        binary = masks_to_binary(masks, img.shape)

        binary = np.where(binary > 127, 255, 0).astype(np.uint8)
        assert len(np.unique(binary)) <= 2, "La máscara no quedó binaria."

        base_name = os.path.splitext(os.path.basename(path))[0]
        out_name = f"{base_name}_mask.png"
        cv2.imwrite(os.path.join(OUTPUT_DIR, out_name), binary)

    print(f" {len(image_paths)} máscaras binarias guardadas en:")
    print(f"   {OUTPUT_DIR}")
else:
    print("  Sin imágenes: no hay máscaras que exportar.")

 20 máscaras binarias guardadas en:
   /content/drive/MyDrive/Under_Water_Dataset/Dataset_Processed/SAM_Masks_Frames_CLAHE
